# Project 05: Diffusion-Inspired Image Denoising Baseline

Train a tiny noise-prediction model on real CIFAR-10 images and monitor learning with NeuroGebra Observatory modules.


## Dataset Source and Download Instructions

- Dataset: CIFAR-10 (Krizhevsky, University of Toronto)
- Official source: https://www.cs.toronto.edu/~kriz/cifar.html
- License: dataset terms from source
- Auto-download in this notebook: `torchvision.datasets.CIFAR10(download=True)`
- Manual fallback:
  1. Download archive from the official source.
  2. Extract under `./data/cifar-10-batches-py`.
  3. Re-run data loading.


In [ ]:
%pip -q install "neurogebra[logging]" torch torchvision matplotlib


In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

from neurogebra.logging.epoch_summary import EpochSummarizer
from neurogebra.logging.logger import LogLevel, TrainingLogger

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

ds = datasets.CIFAR10(root=Path("./data"), train=True, download=True, transform=transform)
ds = Subset(ds, np.arange(12000))
loader = DataLoader(ds, batch_size=128, shuffle=True, drop_last=True)


In [ ]:
class TinyDenoiser(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(2, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 1, 3, padding=1),
        )

    def forward(self, x):
        return self.net(x)

model = TinyDenoiser().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

T = 200
betas = torch.linspace(1e-4, 0.02, T, device=device)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

logger = TrainingLogger(level=LogLevel.DETAILED)
summarizer = EpochSummarizer()

epochs = 3
logger.on_train_start(total_epochs=epochs, total_samples=len(loader.dataset), batch_size=loader.batch_size)

epoch_losses = []
for epoch in range(epochs):
    logger.on_epoch_start(epoch)
    running_loss = 0.0

    for batch_idx, (x0, _) in enumerate(loader):
        x0 = x0.to(device)
        bs, _, h, w = x0.shape

        t = torch.randint(0, T, (bs,), device=device)
        a_bar = alpha_bars[t].view(bs, 1, 1, 1)
        noise = torch.randn_like(x0)
        xt = torch.sqrt(a_bar) * x0 + torch.sqrt(1.0 - a_bar) * noise

        t_channel = (t.float() / float(T - 1)).view(bs, 1, 1, 1).expand(-1, 1, h, w)
        model_input = torch.cat([xt, t_channel], dim=1)

        optimizer.zero_grad()
        pred_noise = model(model_input)
        loss = criterion(pred_noise, noise)
        loss.backward()
        optimizer.step()

        batch_loss = float(loss.item())
        logger.on_batch_end(batch_idx, epoch=epoch, loss=batch_loss)
        running_loss += batch_loss

        grad_norms = {}
        for name, p in model.named_parameters():
            if p.grad is not None:
                grad_norms[name] = float(p.grad.data.norm().item())

        summarizer.record_batch(
            epoch=epoch,
            metrics={"loss": batch_loss},
            gradient_norms=grad_norms,
        )

    avg_loss = running_loss / len(loader)
    epoch_losses.append(avg_loss)
    print(summarizer.finalize_epoch(epoch).format_text())
    logger.on_epoch_end(epoch, metrics={"loss": avg_loss, "accuracy": 0.0, "val_loss": avg_loss, "val_accuracy": 0.0})

logger.on_train_end(final_metrics={"loss": epoch_losses[-1]})


In [ ]:
model.eval()
with torch.no_grad():
    x0, _ = next(iter(loader))
    x0 = x0[:6].to(device)
    bs, _, h, w = x0.shape
    t = torch.full((bs,), 150, device=device, dtype=torch.long)
    a_bar = alpha_bars[t].view(bs, 1, 1, 1)
    noise = torch.randn_like(x0)
    xt = torch.sqrt(a_bar) * x0 + torch.sqrt(1.0 - a_bar) * noise

    t_channel = (t.float() / float(T - 1)).view(bs, 1, 1, 1).expand(-1, 1, h, w)
    pred_noise = model(torch.cat([xt, t_channel], dim=1))
    x0_hat = (xt - torch.sqrt(1.0 - a_bar) * pred_noise) / torch.sqrt(a_bar)

x0_np = x0.cpu().numpy()
xt_np = xt.cpu().numpy()
xhat_np = x0_hat.cpu().numpy()

fig, axes = plt.subplots(3, 6, figsize=(12, 6))
for i in range(6):
    axes[0, i].imshow(x0_np[i, 0], cmap="gray")
    axes[1, i].imshow(xt_np[i, 0], cmap="gray")
    axes[2, i].imshow(xhat_np[i, 0], cmap="gray")
    for r in range(3):
        axes[r, i].axis("off")
axes[0, 0].set_ylabel("Original")
axes[1, 0].set_ylabel("Noisy")
axes[2, 0].set_ylabel("Denoised")
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 3))
plt.plot(epoch_losses)
plt.title("Diffusion Baseline Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.show()
